# 01 · 데이터 먼저 이해하기 — CholecSeg8k

> **이 노트북은 "공부 기록"입니다.** 코드를 돌리는 게 목적이 아니라
> *데이터가 무엇인지 이해하는 것*이 목적입니다. 모든 코드 블록 앞에는
> "왜 이걸 하는지" 설명이 있습니다. 천천히 읽으세요.

## 이 프로젝트가 뭔가요?

복강경 **담낭절제술**(쓸개를 떼어내는 수술)에서, 외과의는 쓸개로 들어가는
관(담관)을 잘못 잘라 큰 합병증을 내지 않도록, 자르기 전에
**"안전한 시야(Critical View of Safety, CVS)"** 를 확보했는지 확인합니다.
이 프로젝트는 그 확인을 **AI로 자동화**합니다.

전체 그림은 2단계입니다:

```
수술 영상 한 장  ─▶  ① 해부구조 분할  ─▶  ② CVS 3개 기준 판정  ─▶  CVS 점수(0~3)
```

이 노트북(`01`)은 ①을 학습시키기 전에, **그 재료인 데이터를 이해**하는
단계입니다.

## 이 노트북에서 할 일

1. CholecSeg8k 데이터셋을 받아서 **한 장 직접 눈으로 보기**
2. 원본 13개 라벨을 → 우리가 쓸 **6개로 합치는 이유** 이해
3. **"클래스 불균형"** 문제를 그래프로 확인
4. 데이터를 왜 **"영상 단위"로 나눠야** 하는지 (데이터 누수)

GPU는 필요 없습니다. CPU 런타임으로 충분합니다.

## 0. 환경 준비

**왜 필요한가:** Colab은 실행할 때마다 *깨끗한 빈 컴퓨터*입니다. 우리 프로젝트
코드(`src/`)도, 필요한 라이브러리도 처음엔 없습니다. 그래서 맨 먼저
(1) GitHub에서 코드를 내려받고 (2) 라이브러리를 설치합니다.

**무슨 일이 일어나는가:** 아래 셀은 repo를 `/content/surgical-ai`에 복제하고,
`scripts/colab_setup.sh`로 의존성을 설치합니다. 이미 복제돼 있으면 새로 받지
않고 최신 코드만 당겨옵니다 — 그래서 **여러 번 실행해도 안전**합니다.

> 모르면 찾아볼 키워드: `git clone`, `pip install`, Colab 런타임(runtime)

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
!bash scripts/colab_setup.sh
print("준비 완료 ·", os.getcwd())

### (선택) Google Drive 연결 — 다운로드·체크포인트 보존

Colab은 런타임이 초기화되면 `/content`가 비워져 데이터(3GB+)와 학습 체크포인트가
사라집니다. 아래 셀은 Drive를 마운트해 `data/`·`outputs/`·모델 캐시를 Drive에
저장하므로, **다음에 다시 와도 받아둔 데이터와 학습 결과를 그대로** 씁니다.

**기본값은 `False`(Drive 미사용)** — 전부 `/content` 에서 돌아갑니다. Drive 에 보존하고 싶을 때만 `USE_DRIVE = True` 로 바꾸세요(인증 창이 뜹니다).

In [ ]:
USE_DRIVE = False  # Drive 에 데이터·체크포인트를 보존하려면 True (Drive 없으면 그대로 두세요)
if USE_DRIVE:
    from src.utils.colab import mount_drive
    mount_drive()

## 1. CholecSeg8k — 이 데이터가 뭔가요?

**CholecSeg8k**는 실제 담낭절제술 영상에서 뽑은 **8,080장의 프레임**입니다.
그리고 각 프레임의 **모든 픽셀**에 "이 픽셀은 간, 이 픽셀은 쓸개, 이 픽셀은
수술도구"라는 라벨이 칠해져 있습니다.

이렇게 픽셀마다 라벨을 붙이는 일을 **semantic segmentation(의미 분할)**이라고
합니다. 사진 한 장을 한 단어로 분류("이건 강아지 사진")하는 것과 달리,
**픽셀 단위**로 "여기는 무엇"인지 칠하는 것이죠. *색칠 공부 책*을 떠올리면
됩니다 — 각 영역이 "그게 무엇이냐"에 따라 색이 정해진 그림.

8,080장은 **17개의 서로 다른 수술 영상**에서 나왔습니다. (이 사실은 4번
섹션에서 아주 중요해집니다.)

**다운로드:** 아래 셀은 HuggingFace Hub에서 약 3.1GB를 받아 로컬 캐시에
저장합니다. 한 번 받으면 다음부터는 캐시에서 바로 읽으므로 다시 받지 않습니다.

> 키워드: semantic segmentation, CholecSeg8k, Cholec80, HuggingFace datasets

In [ ]:
# Optional: HuggingFace token for higher rate limits + faster downloads.
# Get a token at https://huggingface.co/settings/tokens (read scope is enough),
# then uncomment the two lines below and paste it before running this cell.
# from huggingface_hub import login
# login(token="hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

!bash scripts/download_cholecseg8k.sh

## 2. 데이터 한 장 직접 보기

**왜:** 데이터를 다루기 전에 *눈으로 한 장 보는 것*이 가장 중요합니다.
"입력이 어떻게 생겼고, 정답이 어떻게 생겼는지"를 머릿속에 그려야 합니다.

**무슨 일이 일어나는가:** 우리 프로젝트 코드 `src/data/cholecseg8k.py`의
`load_cholecseg8k()`로 데이터를 불러오고, 첫 번째 프레임의
- `image` — 수술 장면 사진 (**모델의 입력**)
- `color_mask` — 색으로 칠해진 정답 마스크 (**모델이 맞춰야 할 정답**)

을 나란히 그려봅니다.

> 키워드: dataset, 입력/정답(label), PIL 이미지, numpy 배열

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.data.cholecseg8k import load_cholecseg8k

dataset = load_cholecseg8k()
frames = dataset["train"] if hasattr(dataset, "keys") else dataset
print("전체 프레임 수:", len(frames))
print("각 프레임이 가진 정보(컬럼):", frames.column_names)

sample = frames[0]
image = np.asarray(sample["image"].convert("RGB"))
color_mask = np.asarray(sample["color_mask"])
print("이미지 크기:", image.shape, "| 마스크 크기:", color_mask.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(image);      ax[0].set_title("surgical frame (model input)")
ax[1].imshow(color_mask); ax[1].set_title("ground-truth mask (color = anatomy)")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 3. 13개 클래스 → 6개 클래스 — 왜 합치는가

CholecSeg8k의 원본 마스크는 **13가지**를 구분합니다 (배경, 복벽, 간, 위장관,
지방, grasper, 결합조직, 혈액, 담낭관, 전기소작기, 쓸개, 간정맥, 간 인대).

하지만 우리의 목표는 **CVS 평가**이고, CVS를 보려면 다음 **6가지**만 있으면
됩니다:

```
인덱스  6-class 이름        원본 13개 중 무엇을 합쳤나
  0     background      ← 배경, 복벽, 위장관, 지방, 결합조직, 혈액
  1     liver           ← 간, 간정맥, 간 인대
  2     gallbladder     ← 쓸개
  3     cystic_duct     ← 담낭관
  4     cystic_artery   ← (없음 — 아래 설명)
  5     tool            ← grasper, 전기소작기
```

**왜 합치나:** 모델에게 "이 13개를 다 구분해" 라고 시키면, CVS와 무관한
구분(지방 vs 결합조직 등)에 학습 능력을 낭비합니다. 우리가 *진짜 필요한*
6가지에 집중시키는 게 낫습니다. 이렇게 라벨 체계를 다시 정의하는 걸
**class remapping**이라 합니다.

아래 셀은 `src/data/cholecseg8k.py`의 `remap_mask()`로 원본 마스크를
6-class로 바꿔서 보여줍니다.

> 키워드: class remapping, label taxonomy, 도메인 지식

In [ ]:
from src.data.cholecseg8k import remap_mask, CLASS_NAMES

mask6 = remap_mask(color_mask)   # (H, W) — 값이 0~5인 정수 배열
print("6-class 이름:", CLASS_NAMES)
print("이 프레임에 실제 등장한 클래스 인덱스:", sorted(np.unique(mask6).tolist()))

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(image);                               ax[0].set_title("frame")
ax[1].imshow(color_mask);                          ax[1].set_title("raw 13-class mask")
ax[2].imshow(mask6, vmin=0, vmax=5, cmap="tab10"); ax[2].set_title("our 6-class mask")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

### 잠깐 — `cystic_artery`(담낭동맥)는 왜 비어 있나?

위 표에서 인덱스 4 `cystic_artery`는 "원본에 없음"이라고 했습니다.

**CholecSeg8k에는 담낭동맥 라벨이 아예 없습니다.** 그래서 이 데이터로
학습할 때 4번 클래스는 **0픽셀**입니다. 없는 라벨을 억지로 지어내지 않고,
*정직하게* 비워 둡니다. 담낭동맥은 나중에 그것을 라벨한 다른 데이터셋
(Endoscapes2023)에서 배웁니다.

데이터의 한계를 숨기지 않고 드러내는 것 — 이것도 공부의 일부입니다.

## 4. 클래스 불균형 — 왜 중요한가

수술 영상에서 **배경과 간**은 화면의 대부분을 차지하지만, **담낭관**은
아주 작은 영역만 차지합니다. 즉 클래스마다 픽셀 수가 극단적으로 다릅니다.
이걸 **클래스 불균형(class imbalance)**이라고 합니다.

**왜 문제인가 — 비유:** 어떤 시험의 정답이 95%가 "A"라면, 아무 생각 없이
전부 "A"라고 찍는 학생도 95점입니다. 하지만 그 학생은 아무것도 모릅니다.
마찬가지로, 화면의 70%가 배경이라면 **"전부 배경"이라고 칠하는 모델도
정확도 70%**가 나옵니다 — 쓸모는 전혀 없는데도요.

그래서 우리는 (1) 불균형이 얼마나 심한지 **측정**하고, 나중에 학습할 때
(2) 드문 클래스에 **가중치를 더 주는** 방식으로 대응합니다.

아래 셀은 앞쪽 200장의 픽셀을 세어 클래스별 비율을 그립니다.

> 키워드: class imbalance, pixel frequency, loss weighting

In [ ]:
counts = np.zeros(len(CLASS_NAMES), dtype=np.int64)
N = 200
for i in range(min(N, len(frames))):
    m = remap_mask(np.asarray(frames[i]["color_mask"]))
    counts += np.bincount(m.ravel(), minlength=len(CLASS_NAMES))[:len(CLASS_NAMES)]
frac = counts / counts.sum()

plt.figure(figsize=(9, 4))
bars = plt.bar(CLASS_NAMES, frac, color="steelblue")
plt.ylabel("pixel fraction")
plt.title(f"per-class pixel distribution (first {N} frames)")
plt.xticks(rotation=15)
for bar, f in zip(bars, frac):
    plt.text(bar.get_x() + bar.get_width() / 2, f, f"{f:.1%}",
             ha="center", va="bottom")
plt.tight_layout()
plt.show()

for name, f in zip(CLASS_NAMES, frac):
    print(f"  {name:14s} {f:7.2%}")

## 5. 데이터를 "영상 단위"로 나누기 — 가장 중요한 부분

머신러닝은 데이터를 세 묶음으로 나눕니다:

- **train (훈련용)** — 모델이 보고 배우는 데이터
- **val (검증용)**  — 학습 도중 "잘 하고 있나" 확인용
- **test (시험용)** — 학습이 끝난 뒤 진짜 실력을 재는, 모델이 한 번도 못 본 데이터

여기 함정이 있습니다. 8,080장은 17개 **영상**에서 나왔고, 한 영상의 연속된
프레임들은 **거의 똑같이 생겼습니다** (0.04초 사이에 수술 장면이 크게 바뀌지
않으니까요).

만약 프레임을 마구 섞어서 나누면:

```
나쁜 방식 — 프레임을 섞어서 나눔
  video01:  f1    f2    f3    f4    f5    f6   ...
            |     |     |     |     |     |
          train  test  train test  train test
                  ^^^^                ^^^^
        f2는 f1과 거의 같은 그림인데 test로 감 → 모델은 train에서 f1을 봤음
```

→ 모델이 시험(test)에서 보는 그림이 훈련에서 본 것과 사실상 같습니다.
점수가 **부풀려집니다.** 실제로는 못 하는데 잘 하는 것처럼 보이죠.
이걸 **데이터 누수(data leakage)**라고 합니다.

올바른 방식은 **영상을 통째로** 한 묶음에만 넣는 것입니다:

```
좋은 방식 — 영상 단위로 나눔
  video01 ─▶ train   (이 영상의 모든 프레임)
  video02 ─▶ train
   ...
  video16 ─▶ test    (이 영상의 모든 프레임)
```

→ test 영상은 모델이 **단 한 프레임도 본 적이 없습니다.** 정직한 점수죠.

`src/data/cholecseg8k.py`의 `make_video_level_split()`이 17개 영상을
**12 train / 2 val / 3 test**로 나눕니다. 아래 셀은 (1) 개념을 가상의 영상
6개로 먼저 보여주고, (2) 진짜 CholecSeg8k 3개 split의 크기를 출력합니다.

> 키워드: train/val/test split, data leakage, video-level split

In [ ]:
from src.data.cholecseg8k import make_video_level_split, CholecSeg8kDataset

# (1) 개념 확인 — 가상의 영상 6개를 4 / 1 / 1 로 나눠보기
toy_videos = [f"video{i:02d}" for i in range(1, 7)]
toy_split = make_video_level_split(toy_videos, n_train=4, n_val=1, n_test=1)
print("가상 영상 6개:", toy_videos)
for name, group in toy_split.items():
    print(f"  {name:5s} -> {sorted(group)}")

# (2) 진짜 CholecSeg8k 의 3개 split (프레임 수)
print()
print("CholecSeg8k 영상 단위 split:")
for split_name in ("train", "val", "test"):
    n = len(CholecSeg8kDataset(split=split_name))
    print(f"  {split_name:5s}: {n:5d} 프레임")

## 마무리 — 이 노트북 정리

### 이 노트북에서 배운 것
- **CholecSeg8k**는 담낭절제술 프레임 8,080장 + 픽셀 단위 정답 마스크.
  픽셀마다 라벨을 칠하는 일이 **semantic segmentation**.
- 원본 **13개 클래스**를 CVS에 필요한 **6개**로 합쳤다 (class remapping).
  이유: 무관한 구분에 학습력을 낭비하지 않으려고.
- `cystic_artery`는 CholecSeg8k에 라벨이 없어 비어 있다 — 데이터의 한계를
  숨기지 않고 정직하게 둔다.
- **클래스 불균형**: 배경/간이 대부분, 담낭관은 극소수. 안 다루면 "전부
  배경" 모델도 정확도가 높게 나와 속는다.
- **영상 단위 split**: 연속 프레임이 너무 비슷해서, 프레임을 섞어 나누면
  **데이터 누수**가 생긴다. 그래서 영상을 통째로 한 묶음에만 넣는다.

### 아직 이해가 덜 된 것 / 더 파볼 것
- `remap_mask()`는 색(RGB)을 어떻게 클래스 번호로 바꾸나? →
  `src/data/cholecseg8k.py`의 `CHOLECSEG8K_COLOR_MAP`을 열어보기.
- 불균형은 측정만 했다. *어떻게* 대응하나? (loss 가중치, 가중 샘플러) →
  다음 노트북에서 실제로 적용해 본다.
- val과 test는 정확히 무엇이 다른가? (val = 학습 중 점검 / test = 최종 1회)

### 다음 노트북에서 할 것
**`02_baseline_unet.ipynb`** — 여기서 이해한 데이터로 첫 모델인
**U-Net 베이스라인**을 직접 학습시킨다. "분할 모델이 어떻게 학습되는가",
그리고 불균형 대응(가중 loss)을 코드로 적용해 본다.